### 05 - Regression Preparation
#### HumanForYou - Attrition ML

Objectif : preparer le dataset California Housing pour les notebooks de regression.

- **Entree** : `workshops/boucle2/datasets/housing/housing.csv`
- **Sorties** : `data/processed/housing_train_prepared.csv`, `data/processed/housing_test_prepared.csv`


#### 1. Imports et chemins

Cette cellule importe les librairies et definit les chemins de travail utilises dans ce notebook.


In [1]:
# Imports principaux pour le chargement des donnees et le preprocessing.
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


#### 2. Chargement du dataset housing

Cette etape charge le dataset brut California Housing depuis le dossier workshop.


In [2]:
# Chargement du dataset brut et verification de disponibilite du fichier source.
HOUSING_CSV_PATH = os.path.join('..', 'workshops', 'boucle2', 'datasets', 'housing', 'housing.csv')
assert os.path.exists(HOUSING_CSV_PATH), f'Fichier introuvable: {HOUSING_CSV_PATH}'

housing = pd.read_csv(HOUSING_CSV_PATH)
print(f'Shape housing brut: {housing.shape}')
housing.head()


Shape housing brut: (20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


#### 3. Split apprentissage / test (stratifie)

Cette cellule construit un split stratifie sur `median_income` pour conserver une distribution stable entre train et test.


In [3]:
# Creation de la variable de stratification puis separation train/test.
housing_for_split = housing.copy()
housing_for_split['income_cat'] = np.ceil(housing_for_split['median_income'] / 1.5)
housing_for_split['income_cat'] = housing_for_split['income_cat'].clip(upper=5.0)

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in split.split(housing_for_split, housing_for_split['income_cat']):
    train_raw = housing_for_split.loc[train_idx].drop(columns=['income_cat']).reset_index(drop=True)
    test_raw = housing_for_split.loc[test_idx].drop(columns=['income_cat']).reset_index(drop=True)

print(f'Train shape: {train_raw.shape}')
print(f'Test shape: {test_raw.shape}')


Train shape: (16512, 10)
Test shape: (4128, 10)


#### 4. Preprocessing (features numeriques et categorielles)

Ici, on ajoute les variables ratio puis on applique un pipeline complet (imputation, standardisation, one-hot).


In [4]:
# Preparation des features et application d'un pipeline de transformation complet.
def add_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['rooms_per_household'] = out['total_rooms'] / out['households']
    out['population_per_household'] = out['population'] / out['households']
    out['bedrooms_per_room'] = out['total_bedrooms'] / out['total_rooms']
    return out

train_features = add_ratio_features(train_raw.drop(columns=['median_house_value']))
test_features = add_ratio_features(test_raw.drop(columns=['median_house_value']))
train_labels = train_raw['median_house_value'].copy()
test_labels = test_raw['median_house_value'].copy()

num_features = train_features.select_dtypes(include=[np.number]).columns.tolist()
cat_features = ['ocean_proximity']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
])

X_train_prepared = preprocessor.fit_transform(train_features)
X_test_prepared = preprocessor.transform(test_features)
feature_names = preprocessor.get_feature_names_out().tolist()

print(f'Nb features apres preprocessing: {len(feature_names)}')


Nb features apres preprocessing: 16


#### 5. Export des jeux prepares

Cette section exporte les jeux train/test preprocesses dans `data/processed/` pour les notebooks suivants.


In [5]:
# Export des datasets prepares avec la cible pour simplifier les etapes de regression.
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_prepared_df = pd.DataFrame(X_train_prepared, columns=feature_names)
train_prepared_df['median_house_value'] = train_labels.to_numpy()

test_prepared_df = pd.DataFrame(X_test_prepared, columns=feature_names)
test_prepared_df['median_house_value'] = test_labels.to_numpy()

train_output_path = os.path.join(PROCESSED_DIR, 'housing_train_prepared.csv')
test_output_path = os.path.join(PROCESSED_DIR, 'housing_test_prepared.csv')

train_prepared_df.to_csv(train_output_path, index=False)
test_prepared_df.to_csv(test_output_path, index=False)

print(f'Saved: {train_output_path}')
print(f'Saved: {test_output_path}')


Saved: ..\data\processed\housing_train_prepared.csv
Saved: ..\data\processed\housing_test_prepared.csv


#### 6. Validation

Cette cellule verifie les dimensions et l'absence de valeurs manquantes apres transformation.


In [6]:
# Verifications de coh?rence pour confirmer la qualite des exports.
assert train_prepared_df.shape[0] == train_raw.shape[0], 'Mismatch lignes train'
assert test_prepared_df.shape[0] == test_raw.shape[0], 'Mismatch lignes test'
assert train_prepared_df.isna().sum().sum() == 0, 'NaN presents dans train_prepared_df'
assert test_prepared_df.isna().sum().sum() == 0, 'NaN presents dans test_prepared_df'
assert 'median_house_value' in train_prepared_df.columns, 'Cible absente du train'
assert 'median_house_value' in test_prepared_df.columns, 'Cible absente du test'

print('Validation OK pour 05_Regression_Preparation.ipynb')


Validation OK pour 05_Regression_Preparation.ipynb
